In [1]:
from cProfile import label
from pathlib import Path

from typing import Any

from black import Transformer
from bob.connections.light import LightConnection
from bob.connections.occupancy import (
    OccupancyInletSystemConnectionPoint,
    OccupancyOutletSystemConnectionPoint,
)

from bob.core import (
    p223,
    Device,
    get_datagraph,
    bind_model_namespace,
    dump,
    clear,
    quantitykind, 
    unit
)

from bob.devices.hvac.damper import ElectricalActuatedDamper
from bob.devices.hvac.coil import ChilledWaterCoil, HotWaterCoil
from bob.devices.hvac.fan import Fan
from bob.devices.hvac.filter import Filter
from bob.devices.hvac.damper import Window
from bob.devices.lighting.light import Luminaire

from bob.devices.electricity.distribution import SinglePhaseDistributionPanel, SinglePoleCircuitBreaker, TwoPolesCircuitBreaker, TwoPolesMainCircuitBreaker
from bob.property import QuantifiableObservableProperty 

from bob.systems.hvac.airhandlingunit import AirHandlingUnit
from bob.systems.hvac.vav import VAV
from bob.sensor.temperature import AirTemperatureSensor
from bob.sensor.flow import AirFlowSensor
from bob.sensor.light import MovementSensor, OccupancySensor

from bob.space.physical import Building, Floor, Roof, Office, Room, Bathroom, Corridor
from bob.space.hvac import HVACSpace, HVACZone
from bob.space.light import LightingSpace, LightingZone

from bob.systems.functionblock import FunctionBlock

from bob.connections.air import *
from bob.connections.electricity import *

model_name = Path("electricity_notebook").stem
__namespace__ = bind_model_namespace("ex", f"urn:ex/{model_name}/")



In [2]:
# First way to define breaker
#mb = TwoPolesCircuitBreaker(label='MainBreaker', amps=200, voltage=240)
#breaker_1 = SinglePoleCircuitBreaker(label='CB#1', amps=15, voltage=120, comment='Lights')
#breaker_2 = TwoPolesCircuitBreaker(label='CB#2', amps=20, voltage=240, comment='Heater')

In [3]:
# Define breaker in template
dp_config = {
    "params": {
        "label": "My Panel",
        "comment": "Description of my panel",
        "voltage": '120_240',
    },
    "sensors": {},
    "contains": {
        ("MainBreaker", TwoPolesMainCircuitBreaker): {
            "comment": "Main breaker of panel",
            "amps": 200,
            "voltage": '120_240',
        },
        ("CB#1", SinglePoleCircuitBreaker): {
            "comment": "Lights",
            "amps": 15,
            "voltage": 120,
            "bus_bar": "A"
        },
        ("CB#2", TwoPolesCircuitBreaker): {
            "comment": "Heater",
            "amps": 20,
            "voltage": 240,
        },
    },
    # other properties could go there... ?
}
dp = SinglePhaseDistributionPanel(config=dp_config)

In [4]:
hq = Electricity_120V_240V_60HzConnection(label='Hydro-Québec')
hq >> dp['MainBreaker']



<TwoPolesMainCircuitBreaker MainBreaker at urn:ex/electricity_notebook/00001>

In [5]:
dump(filename='electricity.ttl')
clear()

@prefix ex: <urn:ex/electricity_notebook/> .
@prefix p223: <http://data.ashrae.org/proposal_to_standard223#> .
@prefix quantitykind: <http://qudt.org/vocab/quantitykind/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix s223: <http://data.ashrae.org/standard223#> .
@prefix unit: <http://qudt.org/vocab/unit/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
ex:00001 a p223:ElectricalCircuitBreaker,
        p223:TwoPolesMainCircuitBreaker,
        s223:Device ;
    rdfs:label "MainBreaker" ;
    p223:hasMaxRange ex:00002 ;
    s223:hasConnectionPoint ex:00003,
        ex:00004,
        ex:00005,
        ex:00006 ;
    s223:hasProperty ex:00002 ;
    rdfs:comment "Main breaker of panel" .
ex:00002 a s223:ObservableProperty,
        s223:QuantifiableObservableProperty,
        s223:QuantifiableProperty ;
    rdfs:label "Current rating of breaker" ;
    s223:hasQuantityKind quantitykind:ElectricCurrent ;
    s223:hasValue 200.0 ;
    s223:unit unit:A .
ex:00003 a s223:C

In [2]:
from bob.sensor.electricity import create_3phases_meter_sensors
from bob.connections.electricity import Electricity_575V_60Hz

s = create_3phases_meter_sensors(label="Meter#1", measuresMedium=Electricity_575V_60Hz)

In [5]:
s[0].__dict__

{'node': rdflib.term.BNode('Nc4caf88bb6454395b4a75b93147ed87b'),
 'label': 'Meter#1',
 'comment': '',
 'hasRole': None,
 'hasPhysicalLocation': None,
 'hasMeasurementLocation': None,
 'hasMeasurementPrecision': None,
 'hasMeasurementUncertainty': None,
 'hasMaxRange': None,
 'hasMinRange': None,
 'measuresMedium': <Medium at http://data.ashrae.org/standard223#Electricity-575V_60Hz>,
 'measuresSubstance': None,
 'observesProperty': <VoltageAB Meter#1.VoltageAB at N7ed1d239cb5c4599bca521575b5c4c5e>,
 '_connection_points': {}}

In [4]:
from bob.connections.electricity import *
from bob.sensor.electricity import create_3phases_meter_sensors

from bob.core import Device, s223, p223, dump, clear, Node

__namespace__ = p223
from typing import Any

class ThreePhasesElectricalMeter(Device):
    node_type = s223.ElectricMeter
    def __init__(self, **kwargs):
        _measuresMedium = kwargs.pop("measuresMedium")
        _label = kwargs['label']
        _hasMeasurementLocation = kwargs.pop("hasMeasurementLocation") if "hasMeasurementLocation" in kwargs else None
        super().__init__(**kwargs)
        self._sensors = create_3phases_meter_sensors(label=_label, measuresMedium=_measuresMedium, hasMeasurementLocation=_hasMeasurementLocation)
        for each in self._sensors:
            self > each

    def set_hasMeasurementLocation(self, node:Node=None):
        for each in self._sensors:
            each.hasMeasurementLocation = node


    def __getitem__(self, name: str) -> Any:
        for each in self._sensors:
            if each.label == name:
                return each


In [5]:
clear()

meter = ThreePhasesElectricalMeter(label="Meter#1", measuresMedium=Electricity_575V_60Hz)

In [7]:
a = Electricity_575V_60HzConnection(label="A")
meter.set_hasMeasurementLocation(a)

In [28]:
dump()

@prefix p223: <http://data.ashrae.org/proposal_to_standard223#> .
@prefix quantitykind: <http://qudt.org/vocab/quantitykind/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix s223: <http://data.ashrae.org/standard223#> .
@prefix unit: <http://qudt.org/vocab/unit/> .
[] a s223:Device,
        s223:ElectricMeter ;
    rdfs:label "Meter#1" ;
    s223:contains [ a p223:VoltageSensor,
                s223:Device,
                s223:Sensor ;
            rdfs:label "Meter#1" ;
            p223:observesProperty [ a p223:MeasuredProperty,
                        p223:QuantifiableMeasuredProperty,
                        p223:VoltageAC,
                        s223:ObservableProperty,
                        s223:QuantifiableObservableProperty,
                        s223:QuantifiableProperty ;
                    rdfs:label "Meter#1.VoltageAC" ;
                    p223:hasQuantityKind quantitykind:Voltage ;
                    p223:measuresMedium s223:Electricity-575V_60Hz 